# ODYSSEY → CIRRUS level correction (H3)

H3 (#20): ODYSSEY and CIRRUS differ in level, and only 4 days overlap. The ticket compares three variants with
the same criteria: a pan-European bridge, a national bridge, and plain seasonal quantile mapping between the two
products (`docs/method.md`, *Level correction (H3)*). The bridge data are not on this PC yet (S7, #77), so this
notebook starts with what ODYSSEY and CIRRUS alone can show.

1. **Distributions over Belgium**, per month and hour of day, for ODYSSEY (2 km / 15 min) and CIRRUS (1 km /
   5 min, and degraded to 2 km).
2. **The overlap days**, 1–5/7/2024, on paired times: the size and shape of the difference.
3. **Calendar months across years:** can two years of CIRRUS give a seasonal quantile mapping?
4. **Hourly radar rain on the ERA5-Land grid,** ready for the pan-European bridge.
5. **Bridge mapping, tested on synthetic data.**
6. **Plain seasonal quantile mapping on the overlap days.**
7. **Radar against ERA5-Land, month by month,** across the change.
8. **Homogeneity of the bridge:** ERA5-Land against VMM, and the radar against both.
9. **ODYSSEY cleaned as in H2,** against ERA5-Land and VMM.
10. **The three variants on the ERA5-Land grid,** a first comparison with the criteria of `docs/method.md`.
11. **CIRRUS in 2026:** fewer wet hours or weaker rain, and where (#88).

In [ ]:
from opera_ds.paths import DATA_DIR, RESULTS_DIR

import datetime as dt
import glob
import os

import matplotlib.pyplot as plt
import numpy as np

from opera_ds.odim_io import read_odim_composite

OUT = RESULTS_DIR / "h3_level"
OUT.mkdir(parents=True, exist_ok=True)
FIG = DATA_DIR.parent / "docs" / "figures" / "h3-level"
FIG.mkdir(parents=True, exist_ok=True)

## 1. Distributions over Belgium

- **Belgium:** the Natural Earth mask `data/mask_BEL_1km.npy` (30,667 pixels at 1 km). At 2 km a cell is
  Belgian when at least 2 of its 4 pixels are (7,744 cells).
- **Data:** the crops of `docs/data.md`, *Derived datasets*: ODYSSEY `DBZH_QIND` 9/2012–7/2024 and CIRRUS `DBZH`
  7/2024–11/2026. CIRRUS is also degraded to 2 km by `zmean`, the degradation closest to real ODYSSEY (H1,
  `docs/method.md`, *Degraded CIRRUS as a stand-in for ODYSSEY*).
- **Counts per frame**, inside the coverage only: `dry` (undetect), `gt70` (above 70 dBZ, impossible values, H2)
  and a histogram of the other echoes from −10 to 70 dBZ in 0.5 dB bins. No other cleaning: #20 starts with the
  cap only.
- **Kept per month,** per hour of day and per kind of time: index 1 for minutes 00/15/30/45 (the ODYSSEY times),
  0 for the other CIRRUS times.
- **Cost:** about 15 ms per ODYSSEY frame and 20 ms per CIRRUS frame. In one process, all 640,000 frames take
  about 3 h; with 16 processes (script version, 11/11/2026) about 15 min. Months already written are skipped.

In [ ]:
EDGES = np.arange(-10.0, 70.001, 0.5)
CENT = (EDGES[:-1] + EDGES[1:]) / 2
NB = len(EDGES) - 1
MASK = np.load(DATA_DIR / "mask_BEL_1km.npy")
_r, _c = np.where(MASK.any(1))[0], np.where(MASK.any(0))[0]
R0, R1 = _r[0] // 2 * 2, (_r[-1] + 2) // 2 * 2      # even edges: the window maps onto the 2 km grid
C0, C1 = _c[0] // 2 * 2, (_c[-1] + 2) // 2 * 2
M1 = MASK[R0:R1, C0:C1]
M2 = M1.reshape(M1.shape[0] // 2, 2, M1.shape[1] // 2, 2).sum((1, 3)) >= 2
print("Belgian pixels at 1 km:", M1.sum(), " cells at 2 km:", M2.sum())


def empty():
    z = lambda: np.zeros((24, 2), np.int64)  # noqa: E731
    return {"n": z(), "valid": z(), "dry": z(), "gt70": z(), "hist": np.zeros((24, 2, NB), np.int64)}


def add(acc, h, q, z, cov, und, m):
    ok = cov & m
    det = ok & ~und & np.isfinite(z)
    acc["n"][h, q] += 1
    acc["valid"][h, q] += ok.sum()
    acc["dry"][h, q] += (ok & und).sum()
    acc["gt70"][h, q] += (det & (z > 70)).sum()
    acc["hist"][h, q] += np.histogram(np.clip(z[det & (z <= 70)], EDGES[0], EDGES[-1]), EDGES)[0]


def window(f, r0, r1, c0, c1):
    a, b = int(f.where.get("crop_row0", 0)), int(f.where.get("crop_col0", 0))
    s = (slice(r0 - a, r1 - a), slice(c0 - b, c1 - b))
    return f.values[s].astype(np.float64), f.coverage[s], f.undetect[s]


def zmean2(z, cov, und):
    """2 x 2 mean in linear Z, as opera_ds.validate.degrade_frame(how='zmean')."""
    sh = (z.shape[0] // 2, 2, z.shape[1] // 2, 2)
    det = cov & ~und
    c = cov.reshape(sh).all((1, 3))
    u = c & ~det.reshape(sh).any((1, 3))
    lin = np.where(det, np.power(10.0, np.nan_to_num(z) / 10.0), 0.0).reshape(sh).mean((1, 3))
    out = np.zeros(c.shape)
    out[c & ~u] = 10 * np.log10(lin[c & ~u])
    return out, c, u


def month_hist(product, ym):
    path = OUT / f"hist_{product}_{ym}.npz"
    if path.exists():
        return
    y, m = ym.split("-")
    files = sorted(glob.glob(str(DATA_DIR / f"{product}_bel" / y / m / "*.h5")))
    accs = {"native": empty()} if product == "odyssey" else {"native": empty(), "deg2": empty()}
    bad = 0
    for p in files:
        try:
            f = read_odim_composite(p, "DBZH")
        except Exception:  # noqa: BLE001
            bad += 1
            continue
        h, q = f.time.hour, int(f.time.minute % 15 == 0)
        if product == "odyssey":
            add(accs["native"], h, q, *window(f, R0 // 2, R1 // 2, C0 // 2, C1 // 2), M2)
        else:
            z, cov, und = window(f, R0, R1, C0, C1)
            add(accs["native"], h, q, z, cov, und, M1)
            add(accs["deg2"], h, q, *zmean2(z, cov, und), M2)
    flat = {f"{k}_{n}": v for k, a in accs.items() for n, v in a.items()}
    np.savez_compressed(path, edges=EDGES, n_files=len(files), n_bad=bad, **flat)


def months_on_disk(product):
    root = DATA_DIR / f"{product}_bel"
    return sorted(f"{os.path.basename(os.path.dirname(d))}-{os.path.basename(d)}"
                  for d in glob.glob(str(root / "*" / "*")) if os.path.isdir(d))


for product in ("odyssey", "cirrus"):
    for ym in months_on_disk(product):
        month_hist(product, ym)
print(len(glob.glob(str(OUT / "hist_odyssey_20*.npz"))), "ODYSSEY months,",
      len(glob.glob(str(OUT / "hist_cirrus_20*.npz"))), "CIRRUS months")

In [ ]:
def load(product, key="native", quarter=None):
    """{YYYY-MM: (histogram, valid pixels)} summed over hours; `quarter` 1 keeps only the ODYSSEY times."""
    out = {}
    for f in sorted(glob.glob(str(OUT / f"hist_{product}_20*.npz"))):
        z = np.load(f)
        h, v = z[f"{key}_hist"], z[f"{key}_valid"]
        h, v = (h[:, quarter], v[:, quarter]) if quarter is not None else (h.sum(1), v.sum(1))
        out[os.path.basename(f)[-11:-4]] = (h.sum(0), v.sum())
    return out


def wet(h, v, thr=10.0):
    """Share of the valid pixels with an echo of at least `thr` dBZ."""
    return h[CENT >= thr].sum() / max(v, 1)


def quant(h, p):
    """Quantile `p` of the echoes (bin centre)."""
    c = np.cumsum(h) / max(h.sum(), 1)
    return CENT[min(np.searchsorted(c, p), NB - 1)]


ODY = load("odyssey")
CIR = load("cirrus", "deg2", quarter=1)       # 2 km, ODYSSEY times
CIR1 = load("cirrus", "native")               # 1 km, all times

## 2. The overlap days, 1–5/7/2024

ODYSSEY at NT is paired with CIRRUS at NT+5, the closest window (`docs/data.md`, *Time windows*). Only cells
covered by both products count. Statistics are on all echoes, as in `docs/method.md`, *Degraded CIRRUS as a
stand-in for ODYSSEY*, and on echoes of at least 10 dBZ.

In [ ]:
cir_files = {os.path.basename(p).split("@")[1]: p for p in glob.glob(str(DATA_DIR / "cirrus_bel" / "2024" / "07" / "*.h5"))}
pair_o, pair_c = empty(), empty()
n_pairs = 0
for p in sorted(glob.glob(str(DATA_DIR / "odyssey_bel" / "2024" / "07" / "*.h5"))):
    t = dt.datetime.strptime(os.path.basename(p).split("@")[1], "%Y%m%dT%H%M")
    k = (t + dt.timedelta(minutes=5)).strftime("%Y%m%dT%H%M")
    if k not in cir_files:
        continue
    zo = window(read_odim_composite(p, "DBZH"), R0 // 2, R1 // 2, C0 // 2, C1 // 2)
    zd = zmean2(*window(read_odim_composite(cir_files[k], "DBZH"), R0, R1, C0, C1))
    both = M2 & zo[1] & zd[1]
    add(pair_o, 0, 0, *zo, both)
    add(pair_c, 0, 0, *zd, both)
    n_pairs += 1
ho, vo = pair_o["hist"][0, 0], pair_o["valid"][0, 0]
hc, vc = pair_c["hist"][0, 0], pair_c["valid"][0, 0]
print(n_pairs, "pairs")
print("share of pixels with echoes >= 10 dBZ: ODYSSEY", round(wet(ho, vo), 3), " CIRRUS", round(wet(hc, vc), 3))
P = [0.1, 0.25, 0.5, 0.75, 0.9, 0.99]
for name, sel in (("all echoes", CENT > -99), ("echoes >= 10 dBZ", CENT >= 10)):
    qo = [quant(np.where(sel, ho, 0), p) for p in P]
    qc = [quant(np.where(sel, hc, 0), p) for p in P]
    print(f"{name:17s} quantiles {P}")
    print("   ODYSSEY       ", qo)
    print("   CIRRUS 2 km   ", qc)
    print("   CIRRUS−ODYSSEY", [round(b - a, 1) for a, b in zip(qo, qc)])

## 3. Calendar months across years

A plain seasonal quantile mapping would fit, for each month of the year, the ODYSSEY distribution onto the CIRRUS
one. ODYSSEY is taken from its last period (8/2018–6/2024, `docs/method.md`, *Radars over time*) and CIRRUS from
8/2024–11/2026 at 2 km on the ODYSSEY times; July 2024 is left out of both, since it holds the overlap days. The
question is whether the product difference stands out of the year-to-year spread of each month.

In [ ]:
def monthly(series, lo, hi, skip=()):
    out = {}
    for ym, (h, v) in series.items():
        if lo <= ym <= hi and ym not in skip:
            out.setdefault(int(ym[5:]), []).append((ym, h, v))
    return out


mo = monthly(ODY, "2018-08", "2024-06")
mc = monthly(CIR, "2024-08", "2026-11")
print("month  ODYSSEY: years, share >= 10 dBZ, median (min-max of the years) | CIRRUS: years, share, median per year")
for m in range(1, 13):
    yo = [(wet(h, v), quant(h, 0.5)) for _, h, v in mo[m]]
    yc = [(wet(h, v), quant(h, 0.5)) for _, h, v in mc[m]]
    so, sc = [a for a, _ in yo], [a for a, _ in yc]
    qo, qc = [b for _, b in yo], [b for _, b in yc]
    print(f"{m:2d}   {len(yo)}  {min(so):.3f}-{max(so):.3f}  {min(qo):5.2f}-{max(qo):5.2f} dBZ | "
          f"{len(yc)}  {', '.join(f'{s:.3f}' for s in sc)}  {', '.join(f'{q:5.2f}' for q in qc)} dBZ")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.4), constrained_layout=True)
for ax, (stat, label) in zip(axes, ((lambda h, v: wet(h, v), "share of Belgian pixels with echoes ≥ 10 dBZ"),
                                    (lambda h, v: quant(h, 0.5), "median of the echoes (dBZ)"))):
    for m in range(1, 13):
        so_ = [stat(h, v) for _, h, v in mo[m]]
        sc_ = [stat(h, v) for _, h, v in mc[m]]
        ax.plot([m - 0.12] * len(so_), so_, "o", ms=4, color="tab:blue", alpha=0.6,
                label="ODYSSEY, one dot per year 8/2018–6/2024" if m == 1 else None)
        ax.plot([m + 0.12] * len(sc_), sc_, "s", ms=5, color="tab:red",
                label="CIRRUS 2 km, one dot per year 8/2024–11/2026" if m == 1 else None)
    ax.set_xticks(range(1, 13))
    ax.set_xlabel("month")
    ax.set_ylabel(label)
    ax.grid(alpha=0.3)
o_d = [wet(ho, vo), quant(ho, 0.5)]
c_d = [wet(hc, vc), quant(hc, 0.5)]
for ax, a, b in zip(axes, o_d, c_d):
    ax.plot(7, a, "*", ms=13, color="tab:blue", mec="k", label="ODYSSEY, 1–5/7/2024 (paired times)")
    ax.plot(7, b, "*", ms=13, color="tab:red", mec="k", label="CIRRUS, 1–5/7/2024 (paired times)")
fig.legend(*axes[0].get_legend_handles_labels(), loc="outside lower center", ncol=2, fontsize=9)
fig.suptitle("ODYSSEY and CIRRUS over Belgium: the year-to-year spread of a month is wider than the product gap")
fig.savefig(FIG / "months-odyssey-cirrus.png", dpi=120)
plt.show()

## 4. Hourly radar rain on the ERA5-Land grid

Ready for the pan-European bridge (S7, #77): the ERA5-Land grid of `scripts/fetch_cds.py`, cell centres at
55.0 … 45.6 N and 3.9 W … 11.8 E, step 0.1° (95 × 158).

- A radar pixel belongs to the cell with the nearest centre (`opera_ds.bridge.cell_index`, as S7); only Belgian
  pixels count. Same cells as the earlier rounding to the 0.1° grid on all Belgian pixels (checked on 1/7/2024).
- **Rain rate:** Marshall–Palmer (Z = 200 R^1.6) from 7 dBZ, capped at 55 dBZ (Imhoff et al. 2020); undetect
  and values above 70 dBZ give no rain.
- **Hours:** each frame stands for the step that ends at its label (5 min CIRRUS, 15 min ODYSSEY), so the hour
  ending at H sums the labels H−55 … H (12 CIRRUS frames) or H−45 … H (4 ODYSSEY frames), as ERA5-Land hourly
  amounts end at their time stamp. The 2.5 min offset of the ODYSSEY window (`docs/data.md`, *Time windows*) is
  ignored. Hours with missing frames are kept with their frame count, to be left out later.
- **Per cell and hour:** the mean rain (mm) of its covered Belgian pixels. Cells with at least 80% of the largest
  pixel count are called full (186 at 2 km, 361 at 1 km).
- **Cost:** about 10 ms per frame (about 2 h in one process; 10 min with 16 processes, 11/11/2026).
- **Caveat:** the composite is a column maximum with no cleaning, so these amounts are an index. In July 2023
  the Belgian mean is about 370 mm, an index of the column maximum rather than rain at the ground; the bridge
  comparison therefore matches distributions, not amounts.

In [ ]:
from opera_ds.bridge import cell_index as bridge_cell_index

LAT_N, LON_W, STEP, NLAT, NLON = 55.0, -3.9, 0.1, 95, 158
STEP_MIN = {"odyssey": 15, "cirrus": 5}
FULL_FRAMES = {"odyssey": 4, "cirrus": 12}


def cell_index(where, r0, r1, c0, c1, m):
    """ERA5-Land cell (flat index) of each pixel centre of the window [r0:r1, c0:c1], nearest cell centre as in
    opera_ds.bridge (S7); -1 if not Belgian."""
    from pyproj import Proj

    p = Proj(where["projdef"])
    sx, sy = float(where["xscale"]), float(where["yscale"])
    xu, yu = p(float(where["UL_lon"]), float(where["UL_lat"]))
    a, b = int(where.get("crop_row0", 0)), int(where.get("crop_col0", 0))
    rr, cc = np.mgrid[r0 - a:r1 - a, c0 - b:c1 - b]
    lon, lat = p(xu + (cc + 0.5) * sx, yu - (rr + 0.5) * sy, inverse=True)
    elat = np.round(LAT_N - STEP * np.arange(NLAT), 4)
    elon = np.round(LON_W + STEP * np.arange(NLON), 4)
    return np.where(m, bridge_cell_index(elat, elon, lat, lon), -1)   # nearest cell centre, as S7


def rain_rate(z, cov, und):
    zz = np.nan_to_num(z, nan=0.0)
    r = np.where((zz >= 7) & (zz <= 70) & ~und, (10 ** (np.minimum(zz, 55.0) / 10) / 200.0) ** (1 / 1.6), 0.0)
    return np.where(cov, r, np.nan)


def month_era5grid(product, ym):
    path = OUT / f"era5grid_{product}_{ym}.npz"
    if path.exists():
        return
    import calendar

    y, m = map(int, ym.split("-"))
    files = sorted(glob.glob(str(DATA_DIR / f"{product}_bel" / f"{y}" / f"{m:02d}" / "*.h5")))
    nh = calendar.monthrange(y, m)[1] * 24
    start = dt.datetime(y, m, 1, 1)                     # the first hour ends at 01:00
    win = (R0 // 2, R1 // 2, C0 // 2, C1 // 2) if product == "odyssey" else (R0, R1, C0, C1)
    mask = M2 if product == "odyssey" else M1
    idx = None
    frames = np.zeros(nh, np.int32)
    step_h = STEP_MIN[product] / 60.0
    for p in files:
        try:
            f = read_odim_composite(p, "DBZH")
        except Exception:  # noqa: BLE001
            continue
        if idx is None:
            idx = cell_index(f.where, *win, mask)
            sel = idx >= 0
            cells, inv = np.unique(idx[sel], return_inverse=True)
            npix = np.bincount(inv, minlength=len(cells))
            rain_h = np.zeros((nh, len(cells)))
            cover_h = np.zeros((nh, len(cells)), np.int64)
        k = int(np.ceil((f.time - start).total_seconds() / 3600.0))
        if not 0 <= k < nh:
            continue
        a, b = int(f.where.get("crop_row0", 0)), int(f.where.get("crop_col0", 0))
        s = (slice(win[0] - a, win[1] - a), slice(win[2] - b, win[3] - b))
        r = rain_rate(f.values[s].astype(np.float64), f.coverage[s], f.undetect[s])[sel]
        ok = np.isfinite(r)
        rain_h[k] += np.bincount(inv[ok], weights=r[ok] * step_h, minlength=len(cells))
        cover_h[k] += np.bincount(inv[ok], minlength=len(cells))
        frames[k] += 1
    if idx is None:
        return
    mean_rain = rain_h * frames[:, None] / np.maximum(cover_h, 1)
    hours = [(start + dt.timedelta(hours=k)).isoformat() for k in range(nh)]
    np.savez_compressed(path, rain=mean_rain.astype(np.float32), frames=frames, cover=cover_h, npix=npix,
                        cells=np.stack([cells // NLON, cells % NLON], 1), hour_end=np.array(hours))


for product in ("odyssey", "cirrus"):
    for ym in months_on_disk(product):
        month_era5grid(product, ym)

In [ ]:
series = {}
for product in ("odyssey", "cirrus"):
    for f in sorted(glob.glob(str(OUT / f"era5grid_{product}_20*.npz"))):
        z = np.load(f)
        ok = z["frames"] == FULL_FRAMES[product]
        if ok.mean() < 0.5:                        # e.g. ODYSSEY 7/2024 (5 days), CIRRUS 10/2024 (gap)
            continue
        full = z["npix"] >= 0.8 * z["npix"].max()
        r = z["rain"][ok][:, full]
        ym = os.path.basename(f)[-11:-4]
        series[(product, ym)] = (r.mean() * 24 * 30.44, (r >= 0.1).mean())

fig, axes = plt.subplots(2, 1, figsize=(13, 6.5), sharex=True, constrained_layout=True)
for product, colour in (("odyssey", "tab:blue"), ("cirrus", "tab:red")):
    keys = sorted(k for k in series if k[0] == product)
    t = [dt.datetime.strptime(k[1], "%Y-%m") for k in keys]
    axes[0].plot(t, [series[k][0] for k in keys], ".-", color=colour, lw=0.8, label=product.upper())
    axes[1].plot(t, [series[k][1] for k in keys], ".-", color=colour, lw=0.8)
for ax in axes:
    for s in (dt.datetime(2015, 12, 1), dt.datetime(2018, 8, 1), dt.datetime(2024, 7, 1)):
        ax.axvline(s, color="0.4", ls="--", lw=0.8)
    ax.grid(alpha=0.3)
axes[0].set_ylabel("radar rain index\n(mm per 30.44 days)")
axes[1].set_ylabel("share of cell-hours\nwith ≥ 0.1 mm")
axes[0].legend(loc="upper left", fontsize=9)
fig.suptitle("Belgium on the ERA5-Land grid, complete hours only; dashed lines: OPERA changes of 12/2015, 8/2018 and 7/2024")
fig.savefig(FIG / "monthly-radar-era5grid.png", dpi=120)
plt.show()

## 5. Bridge mapping, tested on synthetic data

How a bridge removes the product gap even when the weather differs between the two periods.

1. **ODYSSEY → bridge:** each ODYSSEY value goes to the bridge value with the same cumulative probability,
   both taken over the ODYSSEY years.
2. **Bridge → CIRRUS:** that bridge value goes to the CIRRUS value with the same cumulative probability, both
   taken over the CIRRUS years.
3. **Dry stays dry:** a dry ODYSSEY hour stays dry. The wet frequency is matched through the threshold pair
   (Rosenfeld et al. 1993, section 5a), not by adding rain at random as in the frequency adaptation of Themeßl
   et al. (2012): the correction must stay deterministic.

The same weather series feeds the bridge and a product in each period. ODYSSEY reads half as strong as CIRRUS
and misses weak rain; the truth is what CIRRUS would have seen in the ODYSSEY years. Scores are taken above
the level ODYSSEY still detects, since rain it never sees cannot be restored.

In [ ]:
def prob_map(x, src, dst):
    """Value of `dst` with the same cumulative probability that `x` has in `src` (empirical quantile mapping)."""
    s = np.sort(src)
    p = np.searchsorted(s, x, side="right") / len(s)
    return np.quantile(np.sort(dst), np.clip(p, 0, 1))


def bridge_map(x, src, bridge_src, dst, bridge_dst):
    """src -> bridge level (src years), bridge level -> dst (dst years); dry stays dry."""
    level = prob_map(x, src, bridge_src)
    level[x <= 0] = 0.0
    out = prob_map(level, bridge_dst, dst)
    out[level <= 0] = 0.0
    return out


rng = np.random.default_rng(1)


def weather(n, wet_share, scale):
    return np.where(rng.random(n) < wet_share, rng.gamma(0.7, scale, n), 0.0)


def product(b, gain, floor):
    r = b * gain
    r[r < floor] = 0.0
    return r


n, level = 400_000, 1.2                      # level: what ODYSSEY still detects, in CIRRUS units
print("weather of the CIRRUS years | share >= level: truth, plain, bridge | quantiles 50/90/99 over truth: plain, bridge")
for wet_share, scale in ((0.20, 1.5), (0.12, 1.0), (0.28, 2.0)):
    b_ody, b_cir = weather(n, 0.20, 1.5), weather(n, wet_share, scale)
    ody, cir, truth = product(b_ody, 1.0, 0.6), product(b_cir, 2.0, 0.1), product(b_ody, 2.0, 0.1)
    plain = prob_map(ody, ody, cir)
    plain[ody <= 0] = 0.0
    bridged = bridge_map(ody, ody, b_ody, cir, b_cir)
    share = lambda a: (a >= level).mean()                                   # noqa: E731
    q = lambda a: np.quantile(a[a >= level], [0.5, 0.9, 0.99])               # noqa: E731
    print(f"wet {wet_share:.2f}, scale {scale}: {share(truth):.3f} {share(plain):.3f} {share(bridged):.3f} | "
          f"{np.round(q(plain) / q(truth), 2)} {np.round(q(bridged) / q(truth), 2)}")

## 6. Plain seasonal quantile mapping on the overlap days

The third variant, scored with the first test of `docs/method.md`, *Evaluation criteria*. The ODYSSEY
distribution of the fitting months (period 8/2018–6/2024, all pixels, dry included) is mapped onto the CIRRUS
distribution of the same calendar months (2 km, ODYSSEY times). Dry stays dry. July 2024, which holds the
overlap days, is left out of both. The mapping is then applied to ODYSSEY on the 382 paired times of section 2.

In [ ]:
VALS = np.concatenate([[-99.0], CENT])            # index 0: dry


def dist(series_key, product, key, quarter, months):
    h = np.zeros(NB + 1)
    for ym in months:
        z = np.load(OUT / f"hist_{product}_{ym}.npz")
        hh, dd = z[f"{key}_hist"], z[f"{key}_dry"]
        hh, dd = (hh[:, quarter], dd[:, quarter]) if quarter is not None else (hh.sum(1), dd.sum(1))
        h += np.concatenate([[dd.sum()], hh.sum(0)])
    return h


def plain_mapping(src, dst):
    """For each source bin (dry first), the target value at the same mid-bin cumulative probability."""
    cs, cd = np.cumsum(src) / src.sum(), np.cumsum(dst) / dst.sum()
    mid = cs - src / src.sum() / 2
    out = VALS[np.minimum(np.searchsorted(cd, mid), len(VALS) - 1)]
    out[0] = -99.0                                 # dry stays dry
    return out


def scores(values, weights):
    tot = weights.sum()
    share = [weights[values >= t].sum() / tot for t in (10, 20, 30)]
    det = values > -99
    o = np.argsort(values[det])
    v, w = values[det][o], weights[det][o]
    c = np.cumsum(w) / w.sum()
    return share, [v[np.searchsorted(c, p)] for p in (0.25, 0.5, 0.75, 0.9, 0.99)]


ov_o = np.concatenate([[pair_o["dry"][0, 0]], ho])
ov_c = np.concatenate([[pair_c["dry"][0, 0]], hc])
ref = scores(VALS, ov_c)
rows = [("CIRRUS (target)", ref), ("ODYSSEY, uncorrected", scores(VALS, ov_o))]
ody_july = [f"{y}-07" for y in range(2019, 2024)]
fits = {
    "July, CIRRUS 2025 and 2026": (ody_july, ["2025-07", "2026-07"]),
    "July, CIRRUS 2025 only": (ody_july, ["2025-07"]),
    "June-August, all CIRRUS": ([f"{y}-{m:02d}" for y in range(2019, 2024) for m in (6, 7, 8)],
                                ["2024-08", "2025-06", "2025-07", "2025-08", "2026-06", "2026-07", "2026-08"]),
}
for name, (om, cm) in fits.items():
    mp = plain_mapping(dist(None, "odyssey", "native", None, om), dist(None, "cirrus", "deg2", 1, cm))
    rows.append((f"plain mapping, {name}", scores(mp, ov_o)))
print(f"{'':42s} share >= 10/20/30 dBZ      CIRRUS minus row at q25/50/75/90/99 (dB)")
for name, (share, q) in rows:
    print(f"{name:42s} {share[0]:.3f} {share[1]:.3f} {share[2]:.3f}    "
          + " ".join(f"{a - b:+5.1f}" for a, b in zip(ref[1], q)))

## 7. Radar against ERA5-Land, month by month

First use of the bridge (S9, #83: `data/era5land_bel.zarr`). For every month, the radar rain index of section 4
and ERA5-Land are summed over the same hours and cells, and their ratio is a monthly series across the
ODYSSEY → CIRRUS change (`docs/method.md`, *Homogeneity of the bridge*).

- **Cells:** the 186 ERA5-Land cells that are full for both products (section 4: at least 80% of the largest
  pixel count), so ODYSSEY and CIRRUS are compared on the same area.
- **Hours:** complete radar hours only (4 ODYSSEY or 12 CIRRUS frames) and final ERA5-Land hours only
  (`expver` 1, so the series stops at 8/2026); months with less than half of their hours are left out.
- **Ratio:** total radar index over total ERA5-Land on those cells and hours (`opera_ds.homog.area_ratio`), and
  the share of cell-hours with at least 0.1 mm on each side.
- **Steps:** `opera_ds.homog.step` at 7/2024 (12 and 24 months each side) and at 9/2024; at 12/2015 and 8/2018,
  the dates of known OPERA changes, as a check that the test finds a known break.
- **Caveat:** the radar index has no cleaning (section 4), so the ratio is not a rain bias; only its changes
  over time are read here.


In [ ]:
import xarray as xr

from opera_ds.homog import area_ratio, step

era = xr.open_zarr(DATA_DIR / "era5land_bel.zarr", consolidated=False)
expver = era["expver"].load()


def full_cells(product):
    z = np.load(OUT / f"era5grid_{product}_2024-07.npz")
    return {tuple(c) for c in z["cells"][z["npix"] >= 0.8 * z["npix"].max()]}


COMMON = sorted(full_cells("odyssey") & full_cells("cirrus"))
CI, CJ = np.array([c[0] for c in COMMON]), np.array([c[1] for c in COMMON])
print(len(COMMON), "cells full for both products")

ratio_rows = {}
for product in ("odyssey", "cirrus"):
    for f in sorted(glob.glob(str(OUT / f"era5grid_{product}_20*.npz"))):
        ym = os.path.basename(f)[-11:-4]
        y, m = map(int, ym.split("-"))
        if product == "odyssey" and (y, m) >= (2024, 7):
            continue
        z = np.load(f)
        pos = {tuple(c): k for k, c in enumerate(z["cells"])}
        if not all(c in pos for c in COMMON):
            continue
        sel = np.array([pos[c] for c in COMMON])
        ok = z["frames"] == FULL_FRAMES[product]
        hours = np.array(z["hour_end"], dtype="datetime64[ns]")[ok]
        if ok.mean() < 0.5:
            continue
        final = expver.sel(valid_time=hours).values == 1
        if final.mean() < 0.5:
            continue
        e = era["tp"].sel(valid_time=hours[final]).values[:, CI, CJ]
        r = z["rain"][ok][:, sel][final]
        ratio_rows[(y, m)] = dict(product=product, hours=int(final.sum()), of=int(ok.size),
                                  ratio=area_ratio(np.nansum(r, 0), np.nansum(e, 0), np.arange(len(COMMON))),
                                  wet_radar=float(np.nanmean(r >= 0.1)), wet_era5=float(np.nanmean(e >= 0.1)),
                                  era5_mm=float(np.nanmean(np.nansum(e, 0))))
print(len(ratio_rows), "months,", min(ratio_rows), "→", max(ratio_rows))

ratio = {k: v["ratio"] for k, v in ratio_rows.items()}
for at, n in (((2024, 7), 12), ((2024, 7), 24), ((2024, 9), 12), ((2015, 12), 12), ((2018, 8), 12)):
    s = step(ratio, at, n)
    print(f"step at {at[1]}/{at[0]}, {n} months each side: {s['step']:+.3f} log10 ({10 ** s['step']:.2f}x),"
          f" sd before {s['sd_before']:.3f}, t-test p {s['t_p']:.3f}, Mann-Whitney p {s['mw_p']:.3f}")

print("\nmedian ratio per year and product (wet share radar / ERA5-Land)")
for key in sorted({(y, v["product"]) for (y, m), v in ratio_rows.items()}):
    vals = [v for (y, m), v in ratio_rows.items() if (y, v["product"]) == key]
    print(f"  {key[0]} {key[1]:8s} {len(vals):2d} months  ratio {np.median([v['ratio'] for v in vals]):.2f}"
          f"  wet {np.median([v['wet_radar'] / v['wet_era5'] for v in vals]):.2f}")


In [ ]:
keys = sorted(ratio_rows)
t = [dt.datetime(y, m, 15) for y, m in keys]
fig, axes = plt.subplots(2, 1, figsize=(13, 6.5), sharex=True, constrained_layout=True)
for product, colour in (("odyssey", "tab:blue"), ("cirrus", "tab:red")):
    idx = [i for i, k in enumerate(keys) if ratio_rows[k]["product"] == product]
    axes[0].plot([t[i] for i in idx], [ratio_rows[keys[i]]["ratio"] for i in idx], ".-", color=colour, lw=0.8,
                 label=product.upper())
    axes[1].plot([t[i] for i in idx], [ratio_rows[keys[i]]["wet_radar"] / ratio_rows[keys[i]]["wet_era5"]
                                       for i in idx], ".-", color=colour, lw=0.8)
for ax in axes:
    for s in (dt.datetime(2015, 12, 1), dt.datetime(2018, 8, 1), dt.datetime(2024, 7, 1)):
        ax.axvline(s, color="0.4", ls="--", lw=0.8)
    ax.grid(alpha=0.3)
axes[0].set_yscale("log")
axes[0].set_ylabel("radar index / ERA5-Land\n(monthly totals)")
axes[1].set_ylabel("wet share radar /\nwet share ERA5-Land")
axes[0].legend(loc="upper left", fontsize=9)
fig.suptitle("Belgium, 186 ERA5-Land cells, complete radar hours, final ERA5-Land; dashed: OPERA changes 12/2015, 8/2018, 7/2024")
fig.savefig(FIG / "radar-era5-monthly.png", dpi=110)
plt.show()

print("\n2023-2026, month by month")
for k in keys:
    if k[0] >= 2023:
        v = ratio_rows[k]
        print(f"  {k[0]}-{k[1]:02d} {v['product']:8s} hours {v['hours']:3d}/{v['of']:3d}  ratio {v['ratio']:.2f}"
              f"  wet {v['wet_radar']:.3f} / {v['wet_era5']:.3f}  ERA5-Land {v['era5_mm']:.0f} mm")


In [ ]:
# radars of the CIRRUS composites, one file per hour (docs/data.md, *Radars in the CIRRUS composites*); ~2.5 min
import h5py

RADARS = ["behel", "bejab", "bewid", "frave", "frabb", "frtro", "deess", "deneu", "nldhl", "nlhrw", "bezav"]
print("month    hours " + " ".join(f"{r:>5s}" for r in RADARS))
for d in sorted(glob.glob(str(DATA_DIR / "cirrus_bel" / "20*" / "[01]*"))):
    files = sorted(glob.glob(os.path.join(d, "*.h5")))[::12]
    count, n = dict.fromkeys(RADARS, 0), 0
    for f in files:
        try:
            with h5py.File(f) as h:
                nodes = h["how"].attrs.get("nodes")
        except (OSError, KeyError):
            continue
        nodes = nodes.decode() if isinstance(nodes, bytes) else str(nodes)
        n += 1
        for r in RADARS:
            count[r] += r in nodes
    ym = "-".join(os.path.normpath(d).split(os.sep)[-2:])
    print(f"{ym} {n:5d} " + " ".join(f"{count[r] / max(n, 1):5.2f}" for r in RADARS))


## 8. Homogeneity of the bridge: ERA5-Land against VMM, and the radar against both

The test of `docs/method.md`, *Homogeneity of the bridge*. VMM (5 min radar-only intensity, project's copy
17/5/2022 → 11/2026; months with less than half of their hours complete are left out) is summed per hour on the Belgian ERA5-Land cells by `scripts/vmm_hourly.py`
(`results/h3_level/vmm_hourly_era5grid.npz`).

- **VMM / ERA5-Land:** each month, the hours with all 12 VMM rasters and final ERA5-Land, on the Belgian cells
  that VMM covers; step at 7/2024 and 9/2024 with 12 months each side, 18 as a check.
- **Radar / VMM and radar / ERA5-Land on the same hours:** the 186 cells of section 7, hours complete for the
  radar and for VMM and final for ERA5-Land, so the three ratios share their hours and cells.
- **VMM gaps:** a VMM cell not fully covered in an hour is NaN; that cell-hour is left out on every side of a
  ratio (corrected 12/11/2026: the first version left it out of VMM only).


In [ ]:
import calendar

vmm = np.load(OUT / "vmm_hourly_era5grid.npz")
V_HOURS = vmm["hours"].astype("datetime64[ns]")
V_OK = vmm["counts"] == 12
V_CELLS = vmm["bel_cells"]
V_POS = {int(c): k for k, c in enumerate(V_CELLS)}
NL = vmm["lon"].size
assert np.allclose(vmm["lat"], era["latitude"].values) and np.allclose(vmm["lon"], era["longitude"].values)
V_YM = np.array([str(h)[:7] for h in V_HOURS])
print(len(V_CELLS), "Belgian cells for VMM;", int(V_OK.sum()), "of", V_OK.size, "hours complete")

vmm_rows = {}
for ym in sorted(set(V_YM)):
    y, m = map(int, ym.split("-"))
    sel = V_OK & (V_YM == ym)
    hours = V_HOURS[sel]
    if hours.size == 0 or hours[-1] > era["valid_time"].values[-1]:
        continue
    if sel.sum() < 0.5 * calendar.monthrange(y, m)[1] * 24:      # months with less than half of their hours complete
        continue
    final = expver.sel(valid_time=hours).values == 1
    if final.mean() < 0.5:
        continue
    e = era["tp"].sel(valid_time=hours[final]).values[:, V_CELLS // NL, V_CELLS % NL]
    v = vmm["sums"][sel][final]
    both_ok = np.isfinite(v) & np.isfinite(e)        # VMM cells not covered in an hour are NaN: drop them on both sides
    v, e = np.where(both_ok, v, np.nan), np.where(both_ok, e, np.nan)
    row = dict(hours=int(final.sum()), nan_share=float(1 - both_ok.mean()),
               vmm_era5=area_ratio(np.nansum(v, 0), np.nansum(e, 0), np.arange(V_CELLS.size)),
               wet=float((v[both_ok] >= 0.1).mean() / (e[both_ok] >= 0.1).mean()))
    # radar on the same hours and on the common cells of section 7
    product = "odyssey" if (y, m) < (2024, 7) else "cirrus"
    path = OUT / f"era5grid_{product}_{ym}.npz"
    if path.exists():
        z = np.load(path)
        pos = {tuple(c): k for k, c in enumerate(z["cells"])}
        cells = [c for c in COMMON if c[0] * NL + c[1] in V_POS and c in pos]
        flat = np.array([c[0] * NL + c[1] for c in cells])
        rh = np.array(z["hour_end"], dtype="datetime64[ns]")[z["frames"] == FULL_FRAMES[product]]
        both = np.intersect1d(rh, hours[final])
        r = z["rain"][np.isin(np.array(z["hour_end"], dtype="datetime64[ns]"), both)][:, [pos[c] for c in cells]]
        vv = vmm["sums"][np.isin(V_HOURS, both)][:, [V_POS[f] for f in flat]]
        ee = era["tp"].sel(valid_time=both).values[:, flat // NL, flat % NL]
        ok3 = np.isfinite(r) & np.isfinite(vv) & np.isfinite(ee)
        r, vv, ee = (np.where(ok3, a, np.nan) for a in (r, vv, ee))
        k = np.arange(len(cells))
        row.update(product=product, radar_hours=int(both.size),
                   radar_vmm=area_ratio(np.nansum(r, 0), np.nansum(vv, 0), k),
                   radar_era5=area_ratio(np.nansum(r, 0), np.nansum(ee, 0), k))
    vmm_rows[(y, m)] = row
print(len(vmm_rows), "months,", min(vmm_rows), "→", max(vmm_rows))

for name in ("vmm_era5", "wet", "radar_vmm", "radar_era5"):
    series = {k: v[name] for k, v in vmm_rows.items() if name in v}
    for at, n in (((2024, 7), 12), ((2024, 7), 18), ((2024, 7), 24), ((2024, 9), 12), ((2026, 1), 8)):   # 1/2026: read from the data (#88)
        s = step(series, at, n)
        print(f"{name:10s} step at {at[1]}/{at[0]}, {n} months: {s['step']:+.3f} log10 ({10 ** s['step']:.2f}x),"
              f" n {s['n_before']}/{s['n_after']}, sd before {s['sd_before']:.3f},"
              f" t-test p {s['t_p']:.3f}, Mann-Whitney p {s['mw_p']:.3f}")


In [ ]:
keys = sorted(vmm_rows)
t = [dt.datetime(y, m, 15) for y, m in keys]
fig, ax = plt.subplots(figsize=(13, 4.8), constrained_layout=True)
for name, colour, label in (("vmm_era5", "tab:green", "VMM / ERA5-Land"),
                            ("radar_era5", "tab:purple", "radar index / ERA5-Land"),
                            ("radar_vmm", "tab:orange", "radar index / VMM")):
    ax.plot(t, [vmm_rows[k].get(name, np.nan) for k in keys], ".-", color=colour, lw=0.9, label=label)
for s, txt, ha in ((dt.datetime(2024, 7, 1), "ODYSSEY → CIRRUS ", "right"), (dt.datetime(2024, 9, 1), " Jabbeke 9/2024", "left")):
    ax.axvline(s, color="0.4", ls="--", lw=0.8)
    ax.text(s, 0.98, txt, transform=ax.get_xaxis_transform(), va="top", ha=ha, fontsize=8, color="0.3")
ax.set_yscale("log")
ax.grid(alpha=0.3)
ax.set_ylabel("ratio of monthly totals")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.08), ncol=3, fontsize=9)
ax.set_title("Belgium, complete VMM and radar hours, final ERA5-Land, same hours and cells for the three ratios")
fig.savefig(FIG / "vmm-era5-monthly.png", dpi=110)
plt.show()

for k in keys:
    v = vmm_rows[k]
    print(f"{k[0]}-{k[1]:02d} hours {v['hours']:3d}  VMM NaN {v['nan_share']:.2f}  VMM/ERA5 {v['vmm_era5']:.2f}  wet {v['wet']:.2f}  "
          f"{v.get('product', ''):8s} radar/VMM {v.get('radar_vmm', np.nan):.2f}  radar/ERA5 {v.get('radar_era5', np.nan):.2f}")


## 9. ODYSSEY cleaned as in H2, against ERA5-Land and VMM

#20 asks for ODYSSEY cleaned as in H2 (`opera_ds/clean.py`, first version); CIRRUS stays as it is. The hourly
sums of section 4 are redone on the cleaned ODYSSEY crops by `scripts/odyssey_clean_era5grid.py`
(`results/h3_level/era5grid_odyssey_clean_<YYYY-MM>.npz`, 4/2013 → 6/2024): values above 70 dBZ removed, Gabella
on the whole crop, the static mask of the period (4/2013–11/2015, 12/2015–7/2018, 8/2018–6/2024) switched with
the radars listed that day, EURADCLIM fill. The same files hold the raw sums, checked here against section 4.

Then sections 7 and 8 again, with the cleaned ODYSSEY: same cells, hours and steps.

In [ ]:
def radar_file(product, ym, clean):
    if product == "odyssey" and clean:
        z = np.load(OUT / f"era5grid_odyssey_clean_{ym}.npz")
        return {"rain": z["rain_clean"], "rain_raw": z["rain_raw"], "frames": z["frames"], "cells": z["cells"],
                "hour_end": z["hour_end"], "npix": z["npix"]}
    z = np.load(OUT / f"era5grid_{product}_{ym}.npz")
    return {k: z[k] for k in ("rain", "frames", "cells", "hour_end", "npix")}


# raw sums of the cleaning run against section 4
worst = 0.0
for f in sorted(glob.glob(str(OUT / "era5grid_odyssey_clean_20*.npz"))):
    ym = os.path.basename(f)[-11:-4]
    a, b = radar_file("odyssey", ym, True), radar_file("odyssey", ym, False)
    assert (a["cells"] == b["cells"]).all() and (a["frames"] == b["frames"]).all()
    worst = max(worst, float(np.nanmax(np.abs(a["rain_raw"] - b["rain"]))))
print(len(glob.glob(str(OUT / "era5grid_odyssey_clean_20*.npz"))), "cleaned months; raw sums against section 4,"
      f" largest difference {worst:.2e} mm")


def ratio_series(clean):
    rows = {}
    for product in ("odyssey", "cirrus"):
        pattern = "era5grid_odyssey_clean_20*.npz" if product == "odyssey" and clean else f"era5grid_{product}_20*.npz"
        for f in sorted(glob.glob(str(OUT / pattern))):
            ym = os.path.basename(f)[-11:-4]
            y, m = map(int, ym.split("-"))
            if product == "odyssey" and (y, m) >= (2024, 7):
                continue
            z = radar_file(product, ym, clean)
            pos = {tuple(c): k for k, c in enumerate(z["cells"])}
            if not all(c in pos for c in COMMON):
                continue
            sel = np.array([pos[c] for c in COMMON])
            ok = z["frames"] == FULL_FRAMES[product]
            if ok.mean() < 0.5:
                continue
            hours = np.array(z["hour_end"], dtype="datetime64[ns]")[ok]
            final = expver.sel(valid_time=hours).values == 1
            if final.mean() < 0.5:
                continue
            e = era["tp"].sel(valid_time=hours[final]).values[:, CI, CJ]
            r = z["rain"][ok][:, sel][final]
            rows[(y, m)] = dict(product=product, ratio=area_ratio(np.nansum(r, 0), np.nansum(e, 0), np.arange(len(COMMON))),
                                wet=float(np.nanmean(r >= 0.1) / np.nanmean(e >= 0.1)))
    return rows


clean_rows = ratio_series(True)
print(len(clean_rows), "months,", min(clean_rows), "→", max(clean_rows))
for name in ("ratio", "wet"):
    for label, rows in (("raw", ratio_rows), ("cleaned", clean_rows)):
        series = {k: (v[name] if name == "ratio" else v["wet"] if "wet" in v else v["wet_radar"] / v["wet_era5"])
                  for k, v in rows.items()}
        for at, n in (((2024, 7), 12), ((2024, 7), 24), ((2015, 12), 12), ((2018, 8), 12)):
            s = step(series, at, n)
            print(f"{name:5s} {label:7s} step at {at[1]}/{at[0]}, {n} months: {s['step']:+.3f} log10"
                  f" ({10 ** s['step']:.2f}x), sd before {s['sd_before']:.3f}, t-test p {s['t_p']:.3f},"
                  f" Mann-Whitney p {s['mw_p']:.3f}")

print("\nODYSSEY / ERA5-Land per year, median of the months: raw, cleaned")
for y in range(2013, 2025):
    a = [v["ratio"] for (yy, m), v in ratio_rows.items() if yy == y and v["product"] == "odyssey"]
    b = [v["ratio"] for (yy, m), v in clean_rows.items() if yy == y and v["product"] == "odyssey"]
    if a and b:
        print(f"  {y}: {np.median(a):.2f}  {np.median(b):.2f}  ({len(b)} months)")

In [ ]:
# radar / VMM and radar / ERA5-Land on the same hours and cells, with the cleaned ODYSSEY (as section 8)
clean_vmm = {}
for ym in sorted(set(V_YM)):
    y, m = map(int, ym.split("-"))
    product = "odyssey" if (y, m) < (2024, 7) else "cirrus"
    path = OUT / (f"era5grid_odyssey_clean_{ym}.npz" if product == "odyssey" else f"era5grid_cirrus_{ym}.npz")
    sel = V_OK & (V_YM == ym)
    hours = V_HOURS[sel]
    if not path.exists() or hours.size == 0 or hours[-1] > era["valid_time"].values[-1]:
        continue
    final = expver.sel(valid_time=hours).values == 1
    if final.mean() < 0.5:
        continue
    z = radar_file(product, ym, True)
    pos = {tuple(c): k for k, c in enumerate(z["cells"])}
    cells = [c for c in COMMON if c[0] * NL + c[1] in V_POS and c in pos]
    flat = np.array([c[0] * NL + c[1] for c in cells])
    he = np.array(z["hour_end"], dtype="datetime64[ns]")
    both = np.intersect1d(he[z["frames"] == FULL_FRAMES[product]], hours[final])
    r = z["rain"][np.isin(he, both)][:, [pos[c] for c in cells]]
    vv = vmm["sums"][np.isin(V_HOURS, both)][:, [V_POS[f] for f in flat]]
    ee = era["tp"].sel(valid_time=both).values[:, flat // NL, flat % NL]
    ok3 = np.isfinite(r) & np.isfinite(vv) & np.isfinite(ee)
    r, vv, ee = (np.where(ok3, a, np.nan) for a in (r, vv, ee))
    k = np.arange(len(cells))
    clean_vmm[(y, m)] = dict(product=product, radar_vmm=area_ratio(np.nansum(r, 0), np.nansum(vv, 0), k),
                             radar_era5=area_ratio(np.nansum(r, 0), np.nansum(ee, 0), k))
for name in ("radar_vmm", "radar_era5"):
    for label, rows in (("raw", vmm_rows), ("cleaned", clean_vmm)):
        series = {k: v[name] for k, v in rows.items() if name in v}
        for at, n in (((2024, 7), 12), ((2024, 7), 18)):
            s = step(series, at, n)
            print(f"{name:10s} {label:7s} step at {at[1]}/{at[0]}, {n} months: {s['step']:+.3f} log10"
                  f" ({10 ** s['step']:.2f}x), n {s['n_before']}/{s['n_after']}, t-test p {s['t_p']:.3f},"
                  f" Mann-Whitney p {s['mw_p']:.3f}")

keys = sorted(set(ratio_rows) | set(clean_rows))
fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True, constrained_layout=True)
for rows, product, colour, label in ((ratio_rows, "odyssey", "0.6", "ODYSSEY, raw"),
                                     (clean_rows, "odyssey", "tab:blue", "ODYSSEY, cleaned as in H2"),
                                     (clean_rows, "cirrus", "tab:red", "CIRRUS (as it is)")):
    ks = [k for k in keys if k in rows and rows[k]["product"] == product]
    t = [dt.datetime(y, m, 15) for y, m in ks]
    wet = [rows[k]["wet"] if "wet" in rows[k] else rows[k]["wet_radar"] / rows[k]["wet_era5"] for k in ks]
    axes[0].plot(t, [rows[k]["ratio"] for k in ks], ".-", color=colour, lw=0.8, label=label)
    axes[1].plot(t, wet, ".-", color=colour, lw=0.8)
for ax in axes:
    for s in (dt.datetime(2015, 12, 1), dt.datetime(2018, 8, 1), dt.datetime(2024, 7, 1)):
        ax.axvline(s, color="0.4", ls="--", lw=0.8)
    ax.grid(alpha=0.3)
axes[0].set_yscale("log")
axes[0].set_ylabel("radar / ERA5-Land\n(monthly totals)")
axes[1].set_ylabel("wet share radar /\nwet share ERA5-Land")
axes[1].legend(*axes[0].get_legend_handles_labels(), loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=3,
               fontsize=9)
axes[0].set_title("Belgium, 186 ERA5-Land cells, complete radar hours, final ERA5-Land; dashed: OPERA changes 12/2015, 8/2018, 7/2024")
fig.savefig(FIG / "radar-era5-monthly-clean.png", dpi=110)
plt.show()

## 10. The three variants on the ERA5-Land grid

A first comparison of the three variants of #20 with the criteria of `docs/method.md`, *Evaluation criteria*, on
the hourly sums of the 186 common cells (mm per cell and hour), where the bridges live. How a correction fitted
on this grid is then applied to the 2 km / 15 min ODYSSEY pixels is not decided here.

- **Data:** ODYSSEY cleaned as in H2 (section 9), CIRRUS as it is, final ERA5-Land, VMM (section 8). Complete
  radar hours only, except on the overlap days, where CIRRUS has no frames at the quarter hours: there an hour
  with at least 8 of its 12 frames is scaled by 12 / frames.
- **Fit periods:** ODYSSEY 8/2018–6/2024 (its last period; *Radars over time*) for the ERA5-Land bridge and the
  plain mapping, 1/2023–6/2024 for VMM (our copy starts in 2023). CIRRUS 7/2024–12/2025, and 7/2024–8/2026 as a
  check, since CIRRUS drops from 1/2026 (#88). The overlap days, 1–5/7/2024, are left out of every fit.
- **Mapping:** empirical quantile mapping (Gudmundsson et al. 2012), pooled over the cells, with a 31-day window
  centred on each day of the year and pooled over the fit years (Themeßl et al. 2012, section 2.2.1). Bridge
  variants: ODYSSEY → bridge value at the same probability (ODYSSEY years), bridge value → CIRRUS at the same
  probability (CIRRUS years), as in section 5; plain: ODYSSEY → CIRRUS. Dry stays dry, so the wet frequency is
  matched downward only (section 5). For VMM, cell-hours that VMM does not cover are left out of the fit.
- **Test 1, overlap days:** wet share (≥ 0.1 mm) and the ratio CIRRUS / corrected ODYSSEY at the 25th–99th
  percentiles of the wet cell-hours and of the mean, on the hours both products have.
- **Test 2, continuity:** each month, log10 of radar / ERA5-Land at the same percentiles of the wet cell-hours, and
  of the totals; the jump is the mean of the 12 CIRRUS months 7/2024–6/2025 minus the mean of the 12 corrected
  ODYSSEY months 7/2023–6/2024, against the standard deviation of the yearly means of corrected ODYSSEY in
  2019–2023 (the yardstick of *Evaluation criteria*). The ERA5-Land variant is fitted on ERA5-Land, so its test 2
  against ERA5-Land is partly in-sample; the same jumps against VMM (7/2023 on) are an independent check, with no
  yardstick, since VMM starts in 2023.

In [ ]:
Q = (25, 50, 75, 90, 99)
WET = 0.1
OVL = (np.datetime64("2024-07-01T01"), np.datetime64("2024-07-05T06"))
VS = vmm["sums"][:, [V_POS[c[0] * NL + c[1]] for c in COMMON]]          # VMM on the 186 cells


def load_cells(product, first, last, min_frames=None):
    # hours, radar (n, 186), ERA5-Land, VMM (NaN where missing) for radar hours with final ERA5-Land
    pattern = "era5grid_odyssey_clean_*.npz" if product == "odyssey" else "era5grid_cirrus_*.npz"
    H, R = [], []
    for f in sorted(glob.glob(str(OUT / pattern))):
        ym = os.path.basename(f)[-11:-4]
        if not first <= ym <= last:
            continue
        z = np.load(f)
        pos = {tuple(c): k for k, c in enumerate(z["cells"])}
        ok = z["frames"] >= (min_frames or FULL_FRAMES[product])
        r = (z["rain_clean"] if product == "odyssey" else z["rain"])[ok][:, [pos[c] for c in COMMON]]
        H.append(np.array(z["hour_end"], dtype="datetime64[ns]")[ok])
        R.append(r * (FULL_FRAMES[product] / z["frames"][ok])[:, None])
    h, r = np.concatenate(H), np.concatenate(R)
    k = h <= era["valid_time"].values[-1]
    h, r = h[k], r[k]
    k = expver.sel(valid_time=h).values == 1
    h, r = h[k], r[k]
    e = era["tp"].sel(valid_time=h).values[:, CI, CJ]
    vv = np.full_like(r, np.nan)
    i = np.searchsorted(V_HOURS, h)
    inv = (i < V_HOURS.size) & (V_HOURS[np.minimum(i, V_HOURS.size - 1)] == h)
    inv &= V_OK[np.minimum(i, V_HOURS.size - 1)]
    vv[inv] = VS[i[inv]]
    return {"h": h, "r": r, "era5": e, "vmm": vv}


def part(d, a, b, drop_overlap=True):
    m = (d["h"] >= np.datetime64(a)) & (d["h"] < np.datetime64(b))
    if drop_overlap:
        m &= ~((d["h"] >= OVL[0]) & (d["h"] <= OVL[1]))
    return {k: x[m] for k, x in d.items()}


def doy(h):
    return (h.astype("datetime64[D]") - h.astype("datetime64[Y]")).astype(int)


def in_window(h, d, half=15):
    k = np.abs(doy(h) - d)
    return np.minimum(k, 365 - k) <= half


def qmap(x, src, dst):
    p = np.searchsorted(np.sort(src), x, side="right") / len(src)
    return np.quantile(dst, np.clip(p, 0, 1))


def correct(d, fit, bridge):
    # bridge None: plain mapping ODYSSEY -> CIRRUS; else ODYSSEY -> bridge -> CIRRUS (section 5); dry stays dry
    out = np.full_like(d["r"], np.nan)
    days = doy(d["h"])
    for day in np.unique(days):
        rows = days == day
        o, c = fit["ody"], fit["cir"]
        wo, wc = in_window(o["h"], day), in_window(c["h"], day)
        x = d["r"][rows]
        if bridge is None:
            src, dst = o["r"][wo].ravel(), c["r"][wc].ravel()
            y = qmap(x, src[np.isfinite(src)], dst[np.isfinite(dst)])
            y[x <= 0] = 0.0
        else:
            so, bo = o["r"][wo].ravel(), o[bridge][wo].ravel()
            sc, bc = c["r"][wc].ravel(), c[bridge][wc].ravel()
            ko, kc = np.isfinite(so) & np.isfinite(bo), np.isfinite(sc) & np.isfinite(bc)
            level = qmap(x, so[ko], bo[ko])
            level[x <= 0] = 0.0
            y = qmap(level, bc[kc], sc[kc])
            y[level <= 0] = 0.0
        out[rows] = np.where(np.isfinite(x), y, np.nan)
    return out


ody = load_cells("odyssey", "2013-04", "2024-07")
cir = load_cells("cirrus", "2024-07", "2026-08")
cir8 = load_cells("cirrus", "2024-07", "2024-07", min_frames=8)
FITS = {}
for cname, cend in (("CIRRUS to 12/2025", "2026-01-01"), ("CIRRUS to 8/2026", "2026-09-01")):
    c_fit = part(cir, "2024-07-01", cend)
    FITS[("ERA5-Land", cname)] = ("era5", {"ody": part(ody, "2018-08-01", "2024-07-01"), "cir": c_fit})
    FITS[("VMM", cname)] = ("vmm", {"ody": part(ody, "2022-01-01", "2024-07-01"), "cir": c_fit})
    FITS[("plain", cname)] = (None, {"ody": part(ody, "2018-08-01", "2024-07-01"), "cir": c_fit})
print({k: (v[1]["ody"]["h"].size, v[1]["cir"]["h"].size) for k, v in FITS.items()}, "fit hours (ODYSSEY, CIRRUS)")

# test 1: overlap days
ov_o, ov_c = part(ody, "2024-07-01", "2024-07-06", False), part(cir8, "2024-07-01", "2024-07-06", False)
both = np.intersect1d(ov_o["h"], ov_c["h"])
ov_o, ov_c = ({k: x[np.isin(d["h"], both)] for k, x in d.items()} for d in (ov_o, ov_c))
print(both.size, "overlap hours,", both[0], "→", both[-1])


def stats(x):
    x = x[np.isfinite(x)]
    w = x[x >= WET]
    return (x >= WET).mean(), np.percentile(w, Q), x.mean()


ref = stats(ov_c["r"])
test1 = {"CIRRUS (target)": ref, "ODYSSEY cleaned, uncorrected": stats(ov_o["r"])}
for (bridge, cname), (key, fit) in FITS.items():
    test1[f"{bridge}, {cname}"] = stats(correct(ov_o, fit, key))
print(f"{'':36s} wet share   CIRRUS / row at q{Q} and mean")
for name, (s, q, m) in test1.items():
    print(f"{name:36s} {s:.3f}      " + " ".join(f"{a / b:5.2f}" for a, b in zip(ref[1], q)) + f"   {ref[2] / m:5.2f}")

In [ ]:
def monthly(d, x, ref):
    # {(y, m): log10 radar / ref at the percentiles Q of the wet cell-hours, and of the totals}
    ym = d["h"].astype("datetime64[M]")
    out = {}
    for m in np.unique(ym):
        k = ym == m
        a, b = x[k].ravel(), d[ref][k].ravel()
        ok = np.isfinite(a) & np.isfinite(b)
        a, b = a[ok], b[ok]
        if (a >= WET).sum() < 50 or (b >= WET).sum() < 50:
            continue
        lq = np.log10(np.percentile(a[a >= WET], Q) / np.percentile(b[b >= WET], Q))
        out[(int(str(m)[:4]), int(str(m)[5:7]))] = np.append(lq, np.log10(a.sum() / b.sum()))
    return out


o_ev, c_ev = part(ody, "2019-01-01", "2024-07-01"), part(cir, "2024-07-01", "2025-07-01")
CASES = {"ODYSSEY cleaned, uncorrected": o_ev["r"]}
for (bridge, cname), (key, fit) in FITS.items():
    CASES[f"{bridge}, {cname}"] = correct(o_ev, fit, key)
LABELS = [f"q{q}" for q in Q] + ["total"]
test2 = {}
for ref in ("era5", "vmm"):
    after = monthly(c_ev, c_ev["r"], ref)
    after = np.array([v for k, v in after.items() if (2024, 7) <= k <= (2025, 6)])
    print(f"\ntest 2 against {'ERA5-Land' if ref == 'era5' else 'VMM'}: jump in log10"
          + (" / sd of the yearly means 2019-2023 (* jump larger)" if ref == "era5" else ""))
    print(f"{'':36s} " + " ".join(f"{x:>13s}" for x in LABELS))
    for name, x in CASES.items():
        mo = monthly(o_ev, x, ref)
        before = np.array([v for k, v in mo.items() if (2023, 7) <= k <= (2024, 6)])
        jump = after.mean(0) - before.mean(0)
        if ref == "era5":
            sd = np.array([np.mean([v for k, v in mo.items() if k[0] == y], 0) for y in range(2019, 2024)]).std(0, ddof=1)
            test2[name] = (jump, sd)
            cells = " ".join(f"{j:+6.3f}/{s:.3f}{'*' if abs(j) > s else ' '}" for j, s in zip(jump, sd))
        else:
            test2[name + " (VMM)"] = (jump, None)
            cells = " ".join(f"{j:+13.3f}" for j in jump)
        print(f"{name:36s} {cells}  n {len(before)}/{len(after)}")

In [ ]:
# check: the artefact month May 2020 (section 9) out of the ODYSSEY fit; CIRRUS to 12/2025. The VMM fit starts in 2023.
def drop_month(d, ym):
    keep = d["h"].astype("datetime64[M]") != np.datetime64(ym)
    return {k: x[keep] for k, x in d.items()}


c_fit = part(cir, "2024-07-01", "2026-01-01")
o_fit = drop_month(part(ody, "2018-08-01", "2024-07-01"), "2020-05")
CHECK = {"ERA5-Land": ("era5", {"ody": o_fit, "cir": c_fit}), "plain": (None, {"ody": o_fit, "cir": c_fit})}
target = test1["CIRRUS (target)"]
print("test 1, overlap days: wet share, CIRRUS / row at q25-q99 and mean")
for name, (key, fit) in CHECK.items():
    for label, (s, q, m) in ((f"{name}, with May 2020", test1[f"{name}, CIRRUS to 12/2025"]),
                             (f"{name}, without May 2020", stats(correct(ov_o, fit, key)))):
        print(f"  {label:32s} {s:.3f}  " + " ".join(f"{a / b:5.2f}" for a, b in zip(target[1], q)) + f"   {target[2] / m:5.2f}")
for ref_key in ("era5", "vmm"):
    after = monthly(c_ev, c_ev["r"], ref_key)
    after = np.array([v for k, v in after.items() if (2024, 7) <= k <= (2025, 6)])
    print(f"test 2 against {'ERA5-Land' if ref_key == 'era5' else 'VMM'}, jump in log10 at q25-q99 and total")
    for name, (key, fit) in CHECK.items():
        x = correct(o_ev, fit, key)
        mo = monthly(o_ev, x, ref_key)
        before = np.array([v for k, v in mo.items() if (2023, 7) <= k <= (2024, 6)])
        old = test2[f"{name}, CIRRUS to 12/2025" + ("" if ref_key == "era5" else " (VMM)")][0]
        print(f"  {name + ', with May 2020':32s} " + " ".join(f"{j:+.3f}" for j in old))
        print(f"  {name + ', without May 2020':32s} " + " ".join(f"{j:+.3f}" for j in after.mean(0) - before.mean(0)))

## 11. CIRRUS in 2026: fewer wet hours or weaker rain, and where (#88)

CIRRUS falls against both bridges from 1/2026 (section 8). Two checks on the hourly sums of the 186 common cells,
CIRRUS against ERA5-Land and against VMM (uncovered VMM cell-hours left out on both sides), with VMM / ERA5-Land
as the control:

- **Fewer or weaker:** each month, the ratio of wet shares (cell-hours ≥ 0.1 mm), of the mean of the wet cell-hours,
  and of the totals; step at 1/2026 with 8 months each side.
- **Where:** for each cell, log10 of radar / bridge over 1–8/2026 minus the same over 5–12/2025, mapped with the
  radars of the CIRRUS composites near Belgium.
- **When:** the date the CIRRUS production changed (`how/comment` of the crops), and weekly ratios 12 weeks on each
  side of 18/11/2025 (OPERA release 2025b, CIRRUS 1.5.0) and of 18/11/2024 as a seasonal control.

In [ ]:
import json

RADARS_NEAR = ["bejab", "behel", "bewid", "bezav", "nldhl", "nlhrw", "frabb", "frave", "deess"]
DB = {r["odimcode"]: (float(r["longitude"]), float(r["latitude"]))
      for r in json.load(open(DATA_DIR / "opera_radars" / "OPERA_RADARS_DB_20261110.json", encoding="utf-8"))
      if r.get("odimcode") in RADARS_NEAR}


def split_rows(d, ref):
    ym = d["h"].astype("datetime64[M]")
    out = {}
    for m in np.unique(ym):
        k = ym == m
        a, b = d["r"][k], d[ref][k]
        ok = np.isfinite(a) & np.isfinite(b)
        a, b = a[ok], b[ok]
        wa, wb = a >= WET, b >= WET
        out[(int(str(m)[:4]), int(str(m)[5:7]))] = dict(wet=wa.mean() / wb.mean(), intensity=a[wa].mean() / b[wb].mean(),
                                                        total=a.sum() / b.sum())
    return out


c_all = part(cir, "2024-07-01", "2026-09-01")
for ref_key, label in (("era5", "ERA5-Land"), ("vmm", "VMM")):
    rows = split_rows(c_all, ref_key)
    for name in ("wet", "intensity", "total"):
        s = step({k: v[name] for k, v in rows.items()}, (2026, 1), 8)
        print(f"CIRRUS / {label:9s} {name:9s} step at 1/2026, 8 months: {s['step']:+.3f} log10 ({10 ** s['step']:.2f}x),"
              f" sd before {s['sd_before']:.3f}, t-test p {s['t_p']:.3f}, Mann-Whitney p {s['mw_p']:.3f}")
ctrl = {"h": c_all["h"], "r": c_all["vmm"], "era5": c_all["era5"]}
rows = split_rows(ctrl, "era5")
for name in ("wet", "intensity", "total"):
    s = step({k: v[name] for k, v in rows.items()}, (2026, 1), 8)
    print(f"VMM / ERA5-Land    {name:9s} step at 1/2026, 8 months: {s['step']:+.3f} log10 ({10 ** s['step']:.2f}x),"
          f" t-test p {s['t_p']:.3f}")

# per cell: change of radar / bridge from 5-12/2025 to 1-8/2026
b25, b26 = part(cir, "2025-05-01", "2026-01-01"), part(cir, "2026-01-01", "2026-09-01")


def cell_ratio(d, num, den):
    a, b = d[num], d[den]
    ok = np.isfinite(a) & np.isfinite(b)
    return np.where(ok, a, 0).sum(0) / np.where(ok, b, 0).sum(0)


change = {"CIRRUS / ERA5-Land": np.log10(cell_ratio(b26, "r", "era5") / cell_ratio(b25, "r", "era5")),
          "CIRRUS / VMM": np.log10(cell_ratio(b26, "r", "vmm") / cell_ratio(b25, "r", "vmm")),
          "VMM / ERA5-Land (control)": np.log10(cell_ratio(b26, "vmm", "era5") / cell_ratio(b25, "vmm", "era5"))}
lat, lon = era["latitude"].values[CI], era["longitude"].values[CJ]
for name, v in change.items():
    print(f"{name:26s} per-cell change, log10: median {np.median(v):+.3f}, 10th-90th percentile {np.percentile(v, 10):+.3f}"
          f" to {np.percentile(v, 90):+.3f}, cells below -0.1: {(v < -0.1).mean():.0%}")
for code, (x, y) in DB.items():
    dist = np.hypot((lon - x) * np.cos(np.radians(51)), lat - y) * 111.0
    near = dist <= 50
    if near.any():
        print(f"  within 50 km of {code}: {near.sum():3d} cells, CIRRUS / ERA5-Land change median"
              f" {np.median(change['CIRRUS / ERA5-Land'][near]):+.3f}, CIRRUS / VMM {np.median(change['CIRRUS / VMM'][near]):+.3f},"
              f" VMM / ERA5-Land {np.median(change['VMM / ERA5-Land (control)'][near]):+.3f}"
              f" (max {change['VMM / ERA5-Land (control)'][near].max():+.3f})")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5.2), constrained_layout=True)
for ax, (name, v) in zip(axes, change.items()):
    sc = ax.scatter(lon, lat, c=v, cmap="RdBu", vmin=-0.4, vmax=0.4, s=60, marker="s")
    for code, (x, y) in DB.items():
        if 2.0 <= x <= 7.0 and 49.3 <= y <= 52.0:
            ax.plot(x, y, "k^", ms=7)
            ax.annotate(code, (x, y), xytext=(4, 4), textcoords="offset points", fontsize=8)
    ax.set_title(name, fontsize=10)
    ax.set_xlim(2.3, 6.6)
    ax.set_ylim(49.4, 51.8)
    ax.set_aspect(1 / np.cos(np.radians(51)))
    ax.grid(alpha=0.3)
fig.colorbar(sc, ax=axes, shrink=0.8, label="log10 change, 1-8/2026 against 5-12/2025")
fig.suptitle("Belgium, 186 ERA5-Land cells: change of the ratio from 5-12/2025 to 1-8/2026; triangles: radars")
fig.savefig(FIG / "cirrus-2026-change-by-cell.png", dpi=110)
plt.show()

In [ ]:
# when did the CIRRUS production change? how/comment of the crops switches once in 10-11/2025 (binary search)
import h5py


def comment(path):
    with h5py.File(path) as f:
        c = f["how"].attrs.get("comment", b"")
    return (c.decode() if isinstance(c, bytes) else str(c))[:40]


files = sorted(glob.glob(str(DATA_DIR / "cirrus_bel" / "2025" / "1[01]" / "*.h5")))
lo, hi = 0, len(files) - 1
first = comment(files[lo])
while hi - lo > 1:
    mid = (lo + hi) // 2
    lo, hi = (mid, hi) if comment(files[mid]) == first else (lo, mid)
print("how/comment changes between", os.path.basename(files[lo]), "and", os.path.basename(files[hi]))

# weekly ratios around 18/11 (OPERA release 2025b, CIRRUS 1.5.0), and the same date one year before as a seasonal control
for t0 in (np.datetime64("2024-11-18T10"), np.datetime64("2025-11-18T10")):
    d = part(cir, str(t0 - np.timedelta64(84, "D"))[:10], str(t0 + np.timedelta64(84, "D"))[:10])
    week = ((d["h"] - t0) // np.timedelta64(7, "D")).astype(int)
    series = {name: {} for name in ("CIRRUS / ERA5-Land", "CIRRUS / VMM", "VMM / ERA5-Land")}
    for w in np.unique(week):
        k = week == w
        if d["era5"][k].mean() * 24 * 7 < 3:              # weeks with less than 3 mm of ERA5-Land are left out
            continue
        a, b, c = d["r"][k], d["era5"][k], d["vmm"][k]
        m = np.isfinite(c)
        series["CIRRUS / ERA5-Land"][(2000, 100 + w)] = a.sum() / b.sum()
        series["CIRRUS / VMM"][(2000, 100 + w)] = a[m].sum() / c[m].sum()
        series["VMM / ERA5-Land"][(2000, 100 + w)] = c[m].sum() / b[m].sum()
    for name, ser in series.items():
        s = step(ser, (2000, 100), 12)
        print(f"{str(t0)[:10]} {name:20s} 12 weeks each side: {s['step']:+.3f} log10 ({10 ** s['step']:.2f}x),"
              f" n {s['n_before']}/{s['n_after']}, t-test p {s['t_p']:.3f}, Mann-Whitney p {s['mw_p']:.3f}")

In [ ]:
# #88: what the Belgian radars deliver (ORD bucket listing, files of 12 UTC), and the single radar against CIRRUS
# near the radar at the same rainy hours, before and after 18/11/2025 (about 75 MB of volumes, kept in data/opera_pvol/issue88)
from opera_ds.paths import DATA_DIR
OUT_H3 = OUT
"""#88: single radar against CIRRUS near the radar, at the same times, before and after 18/11/2025."""
import datetime as dt
import glob
import os
import urllib.request

import h5py
import numpy as np

from opera_ds.bridge import pixel_lonlat  # noqa: E402
from opera_ds.odim_io import read_odim_composite  # noqa: E402
from opera_ds.ord import BUCKET, list_keys  # noqa: E402

OUT = str(DATA_DIR / "opera_pvol" / "issue88")
R = {"behel": (5.4054, 51.0702), "bewid": (5.5044, 49.9135), "bejab": (3.0641, 51.1919)}
WINDOWS = {"before": ("2025-09-15", "2025-11-18"), "after": ("2025-11-19", "2026-02-01")}
N = 15


def rainy_hours(node):
    """Top-N hours (one per day) by ERA5-Land rain within 50 km of the radar, per window; complete CIRRUS hours."""
    import xarray as xr

    era = xr.open_zarr(DATA_DIR / "era5land_bel.zarr", consolidated=False)
    x, y = R[node]
    out = {}
    for name, (a, b) in WINDOWS.items():
        cand = []
        for f in sorted(glob.glob(str(OUT_H3 / "era5grid_cirrus_20*.npz"))):
            ym = os.path.basename(f)[-11:-4]
            if not a[:7] <= ym <= b[:7]:
                continue
            z = np.load(f)
            lat = 55.0 - 0.1 * z["cells"][:, 0]
            lon = -3.9 + 0.1 * z["cells"][:, 1]
            near = np.hypot((lon - x) * 0.63, lat - y) * 111 <= 50
            hours = np.array(z["hour_end"], dtype="datetime64[ns]")
            ok = (z["frames"] == 12) & (hours >= np.datetime64(a)) & (hours < np.datetime64(b))
            if not ok.any():
                continue
            e = era["tp"].sel(valid_time=hours[ok]).values[:, z["cells"][near, 0], z["cells"][near, 1]].copy(); e = np.nanmean(e, 1)
            cand += [(a_, b_) for a_, b_ in zip(e, hours[ok]) if np.isfinite(a_)]
        cand.sort(key=lambda c: -c[0])
        days, pick = set(), []
        for e, h in cand:
            d = str(h)[:10]
            if d not in days:
                days.add(d)
                pick.append(h.astype("datetime64[m]").astype(dt.datetime))
            if len(pick) == N:
                break
        out[name] = pick
    return out


def volume(node, t):
    os.makedirs(OUT, exist_ok=True)
    keys = [k for k, _ in list_keys(f"{t:%Y/%m/%d}/BE/{node}/PVOL/")
            if f"@{t:%Y%m%dT%H%M}@0.3_" in k and k.endswith("@DBZH.h5")]
    if not keys:
        return None
    key = sorted(keys)[0]
    path = os.path.join(OUT, os.path.basename(key))
    if not os.path.exists(path):
        urllib.request.urlretrieve(f"{BUCKET}/{key}", path)
    return path


def radar_stats(path):
    """Share of bins >= 7 dBZ and mean dBZ of those, 5-50 km ground range, lowest elevation and column max."""
    with h5py.File(path) as h:
        sets = []
        for ds in sorted(k for k in h if k.startswith("dataset")):
            w = dict(h[ds]["where"].attrs)
            wa = dict(h[ds]["data1"]["what"].attrs)
            raw = h[ds]["data1"]["data"][()].astype(float)
            z = raw * wa["gain"] + wa["offset"]
            z[raw == wa["nodata"]] = np.nan
            z[raw == wa["undetect"]] = -32.0
            rng = (w["rstart"] + (np.arange(w["nbins"]) + 0.5) * w["rscale"]) * np.cos(np.radians(w["elangle"]))
            sets.append((w["elangle"], w["nbins"], w["rscale"], z, rng))
    low = min(sets, key=lambda s: s[0])
    same = [s for s in sets if (s[1], s[2]) == (low[1], low[2]) and s[3].shape == low[3].shape]
    ring = (low[4] >= 5000) & (low[4] <= 50000)
    zl = low[3][:, ring]
    zc = np.nanmax(np.stack([s[3][:, ring] for s in same]), 0)
    res = {}
    for name, z in (("low", zl), ("colmax", zc)):
        v = z[np.isfinite(z)]
        wet = v >= 7
        res[name] = (wet.mean(), v[wet].mean() if wet.any() else np.nan)
    return res


def cirrus_stats(node, t):
    p = f"{DATA_DIR}/cirrus_bel/{t:%Y}/{t:%m}/OPERA@{t:%Y%m%dT%H%M}@0@DBZH.h5"
    if not os.path.exists(p):
        return None
    fr = read_odim_composite(p)
    z = fr.to_field(0.0)
    lon, lat = pixel_lonlat(fr.where)
    x, y = R[node]
    d = np.hypot((lon - x) * np.cos(np.radians(y)), lat - y) * 111.2
    v = z[(d >= 5) & (d <= 50) & np.isfinite(z)]
    wet = v >= 7
    return wet.mean(), v[wet].mean() if wet.any() else np.nan

def delivered(day, node):
    return sorted({k.split("@", 2)[2] for k, _ in list_keys(f"{day:%Y/%m/%d}/BE/{node}/") if f"@{day:%Y%m%d}T12" in k})


for node, days in (("bewid", ("2025-12-31", "2026-01-01")), ("behel", ("2026-01-06", "2026-01-07")),
                   ("bejab", ("2025-07-20", "2026-03-10"))):
    for d in days:
        print(node, d, [x for x in delivered(dt.date.fromisoformat(d), node) if "_" in x.split("@")[0]])

from scipy.stats import mannwhitneyu

for node in R:
    rows = {}
    for name, ts in rainy_hours(node).items():
        rows[name] = []
        for t in ts:
            vp, cs = volume(node, t), cirrus_stats(node, t)
            if vp is not None and cs is not None:
                rows[name].append((cs, radar_stats(vp)))
    for key in ("low", "colmax"):
        db = np.array([cs[1] - rs[key][1] for cs, rs in rows["before"]])
        da = np.array([cs[1] - rs[key][1] for cs, rs in rows["after"]])
        b = np.array([cs[0] / rs[key][0] for cs, rs in rows["before"] if rs[key][0] > 0])
        a = np.array([cs[0] / rs[key][0] for cs, rs in rows["after"] if rs[key][0] > 0])
        print(f"{node} {key:6s} hours {len(rows['before'])}/{len(rows['after'])}: wet share CIRRUS / radar {np.median(b):.2f} ->"
              f" {np.median(a):.2f} (p {mannwhitneyu(a, b).pvalue:.3f}); mean wet dBZ CIRRUS - radar {np.nanmedian(db):+.1f} ->"
              f" {np.nanmedian(da):+.1f} (p {mannwhitneyu(da[np.isfinite(da)], db[np.isfinite(db)]).pvalue:.3f})")

## Results so far (12/11/2026)

- **Overlap days.** On the 382 paired times, CIRRUS has echoes of at least 10 dBZ on 22.6% of the Belgian
  pixels, ODYSSEY on 15.6%. Over all echoes, CIRRUS is higher by 1–5 dB depending on the quantile (median +4 dB,
  99th percentile +5 dB). Over echoes of at least 10 dBZ the shift is only 0.5–1.5 dB up to the 90th
  percentile: most of the gap is in **how often** ODYSSEY sees rain, not in how strong it sees it.
- **Calendar months.** Across years, the same month changes much more than the products differ: in June the
  ODYSSEY share of pixels with echoes ≥ 10 dBZ goes from 7.5% (2023) to 25% (2020), and the median from 7 to
  15 dBZ. The CIRRUS months fall inside the ODYSSEY spread, often at its low end (dry spring 2025).
- **Consequence for the third variant.** With two years of CIRRUS, a seasonal quantile mapping between the two
  products would copy the weather of 2024–2026 into the ODYSSEY years instead of removing the product
  difference. This is the reason for a bridge in #20; the plain mapping stays as the reference the bridges must
  beat. On the overlap days it moves ODYSSEY away from CIRRUS when fitted on all CIRRUS Julys (July 2026 is dry),
  and closer when fitted on July 2025 alone: the result depends on the CIRRUS year (section 6).
- **July 2026.** CIRRUS has echoes on only 5% of the Belgian pixel-times (95% undetect), against 17–26% in the
  other summer months; the Belgian radars are listed in `how/nodes`. A dry month, not a data problem: 10.6 mm at
  Uccle against a normal of 76.9 mm (RMI) and 13.8 mm in ERA5-Land over the 186 Belgian cells (section 7).
- **Against ERA5-Land (section 7).** Across 7/2024 the wet share rises by 14–21% relative to ERA5-Land with 12
  or 24 months each side (significant with both); the amount ratio by 23% with 12 months and 7% with 24 (only
  the first significant), much less than the factor 2 of the almost dry overlap days. CIRRUS / ERA5-Land falls
  from 5.0 (7–12/2024) to 2.5 (1–8/2026), not explained by missing radars; VMM will tell whether CIRRUS or
  ERA5-Land moves.
- **Homogeneity of the bridge (section 8).** VMM / ERA5-Land has no step at 7/2024 or 9/2024 (+0.022 and +0.013
  in log10, p 0.50 and 0.69; a step of about 18% would have been found), with the VMM cell-hours not covered left
  out on both sides (corrected 12/11/2026). ODYSSEY → CIRRUS is +15% in amounts against VMM and +22–23% against
  ERA5-Land. From 1/2026 CIRRUS falls by about 30–37% against both, while VMM / ERA5-Land does not: a change in
  CIRRUS, date read from the data, cause not known (#88).
- **ODYSSEY cleaned as in H2 (section 9).** The cleaning lowers ODYSSEY / ERA5-Land by 21–33% of the yearly median
  before 2019 and 7–17% from 2019 (25% in 2020); the yearly medians become 2.8–3.9 instead of 3.5–5.6. So the
  ODYSSEY → CIRRUS gap grows to about +36–37% in amounts against ERA5-Land and +27–29% against VMM, and +24–29%
  in the wet share. The artefact months of spring 2018 and May 2020 shrink but stay. Problem issue on the 2026 drop of
  CIRRUS: #88.
- **The three variants (section 10), on the ERA5-Land grid.** The plain mapping is worse than no correction. Both
  bridges about halve the gap on the overlap days (mean still 1.55–1.68 times lower than CIRRUS; ERA5-Land closest
  on the percentiles, VMM on the wet share). ERA5-Land with CIRRUS to 12/2025 is the only variant with every jump
  across 7/2024 within the year-to-year spread against ERA5-Land (partly in-sample); against VMM its total jump is
  −0.001, its q99 −0.073. With CIRRUS 2026 in the fit (#88) it fails at the top end. Open for G1: how to apply a
  bridge-grid mapping to the pixels, the CIRRUS period, the yardstick.
- **CIRRUS in 2026 (section 11, #88).** Against ERA5-Land, CIRRUS mostly sees fewer wet hours (−21%). The drop is
  local: near Helchteren (−0.21 in log10) and Wideumont (−0.18), not near Jabbeke (−0.02); VMM / ERA5-Land rises
  near Helchteren (+0.23). GeoTIFF and HDF5 crops are identical on 12 rainy times, so the source switch of our
  crops at 1/2026 is not the cause. The production changed on 18/11/2025 (CIRRUS 1.5.0, new search distance per
  radar): CIRRUS / ERA5-Land falls by −0.31 there, −0.12 at the same date of 2024; against VMM both years fall by
  −0.23 (season). Best candidate, not proven.
- **May 2020 (check after section 10).** Left out of the ODYSSEY fit, it changes nothing that matters (< 0.005).
- **VMM 2022 added (12/11/2026).** VMM / ERA5-Land has no step at 7/2024 with 24 months each side either (+0.011,
  p 0.70). The VMM variant, now fitted on ODYSSEY 2022–6/2024, comes a little closer to CIRRUS on the overlap days
  (percentiles 1.20–1.38 instead of 1.23–1.47); the ranking of the variants does not change.